In [102]:
import networkx as nx
import geopandas as gpd
from scipy.spatial.distance import cdist
import numpy as np
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter
from sklearn.neighbors import NearestNeighbors
from pymoo.core.population import Population
import matplotlib.pyplot as plt
from pymoo.operators.mutation.bitflip import BitflipMutation
from sklearn.metrics import pairwise_distances
from shapely.geometry import Point

In [103]:
import geopandas as gpd
import numpy as np

# --- Cargar el archivo ---
gdf_nodos = gpd.read_file("dataset_final.gpkg")

# --- Asegurar CRS métrico ---
if gdf_nodos.crs is None:
    print("⚠️ No hay CRS definido. Asumiendo EPSG:4326 (grados).")
    gdf_nodos = gdf_nodos.set_crs(epsg=4326)

if gdf_nodos.crs.to_epsg() != 3857:
    print(f"🌍 Reproyectando de {gdf_nodos.crs} a EPSG:3857 (metros).")
    gdf_nodos = gdf_nodos.to_crs(epsg=3857)

# --- Asegurar geometrías válidas ---
gdf_nodos = gdf_nodos[
    gdf_nodos.geometry.notnull() &
    (~gdf_nodos.geometry.is_empty) &
    (gdf_nodos.is_valid)
]

if len(gdf_nodos) == 0:
    raise ValueError("❌ El archivo no contiene geometrías válidas.")

# --- Convertir todo a puntos si hay polígonos o líneas ---
geom_types = gdf_nodos.geometry.geom_type.unique()
if not np.all(np.isin(geom_types, ["Point", "MultiPoint"])):
    print(f"🔄 Convirtiendo {geom_types} a centroides (puntos representativos)...")
    gdf_nodos = gdf_nodos.copy()
    gdf_nodos["geometry"] = gdf_nodos.geometry.centroid
    gdf_nodos = gdf_nodos.set_geometry("geometry")
    print("✅ Geometrías convertidas a puntos.")

# --- Crear columnas x, y ---
gdf_nodos["x"] = gdf_nodos.geometry.x
gdf_nodos["y"] = gdf_nodos.geometry.y

# --- Asegurar columna osmid ---
if "osmid" not in gdf_nodos.columns:
    print("⚠️ No se encontró columna 'osmid'. Creando índice numérico temporal.")
    gdf_nodos["osmid"] = np.arange(len(gdf_nodos))

# --- Diagnóstico ---
print("Tipos de geometría:")
print(gdf_nodos.geometry.geom_type.value_counts())

print(f"\n✅ Total de nodos válidos: {len(gdf_nodos)}")
print(f"📏 CRS actual: {gdf_nodos.crs}")
print(f"📍 Ejemplo de coordenadas: {list(zip(gdf_nodos.x[:3], gdf_nodos.y[:3]))}")

# --- (Opcional) Guardar el archivo limpio ---
# gdf_nodos.to_file("dataset_final_limpio.gpkg", driver="GPKG")

print(gdf_nodos.geometry.geom_type.unique())



🔄 Convirtiendo ['Polygon'] a centroides (puntos representativos)...
✅ Geometrías convertidas a puntos.
Tipos de geometría:
Point    488
Name: count, dtype: int64

✅ Total de nodos válidos: 488
📏 CRS actual: EPSG:3857
📍 Ejemplo de coordenadas: [(-13469391.21926975, 4555107.731057582), (-13469373.630790208, 4554582.659481793), (-13469435.758197997, 4556036.497738498)]
['Point']


In [104]:
# ------------------ Funciones de fitness ------------------

def fitness_cantidad_estaciones(estaciones_osmid, factor=1.0):
    return np.float64(len(estaciones_osmid) * factor)

def fitness_prioridad_simple(gdf, estaciones_osmid, peso_flujo=0.7, peso_esquinas=0.3):
    flujo_norm = gdf['flujo_total'] / (gdf['flujo_total'].max() + 1e-6)
    esquinas_norm = gdf['esquinas_en_radio'] / (gdf['esquinas_en_radio'].max() + 1e-6)
    gdf['prioridad'] = peso_flujo * flujo_norm + peso_esquinas * esquinas_norm
    prioridades_estaciones = gdf.loc[gdf['osmid'].isin(estaciones_osmid), 'prioridad']
    if len(prioridades_estaciones) > 0:
        return np.float64(prioridades_estaciones.mean())
    else:
        return np.float64(0.0)

def fitness_cobertura_global(gdf_nodos, estaciones_idx, d_max=50000, penalizacion=1e6):

    # --- Validaciones ---
    if gdf_nodos.empty or 'geometry' not in gdf_nodos:
        raise ValueError("❌ gdf_nodos debe ser un GeoDataFrame con geometría válida.")
    if len(estaciones_idx) == 0:
        raise ValueError("❌ Debe haber al menos una estación.")

    # Extraer coordenadas
    coords = np.vstack((gdf_nodos.geometry.x, gdf_nodos.geometry.y)).T
    estaciones = coords[np.array(estaciones_idx)]

    # --- Distancias nodo-estación ---
    dist_matrix = pairwise_distances(coords, estaciones, metric='euclidean')
    dist_min = dist_matrix.min(axis=1)

    # --- Penalización: si está más lejos que d_max, le sumamos mucho ---
    penalizaciones = np.where(dist_min > d_max, penalizacion, 0.0)

    # --- Fitness global ---
    fitness = np.sum(dist_min + penalizaciones)

    return fitness


In [ ]:
class EVChargingProblem3F(Problem):
    def __init__(self, gdf, d_max, factor_estaciones=1.0,
                 peso_flujo=0.7, peso_esquinas=0.3, penalizacion=10,
                 debug=False):
        self.gdf = gdf.copy()
        self.d_max = d_max
        self.factor_estaciones = factor_estaciones
        self.peso_flujo = peso_flujo
        self.peso_esquinas = peso_esquinas
        self.penalizacion = penalizacion
        self.debug = debug
        self.n_nodos = len(gdf)

        # --- Validaciones de entrada ---
        required_cols = ["osmid", "flujo_total", "esquinas_en_radio"]
        for col in required_cols:
            if col not in gdf.columns:
                raise ValueError(f"❌ Falta la columna requerida '{col}' en el GeoDataFrame.")

        super().__init__(
            n_var=self.n_nodos,
            n_obj=3,
            n_constr=0,
            xl=0, xu=1,
            type_var=int
        )

    def _evaluate(self, X, out, *args, **kwargs):
        f1_raw = np.zeros(len(X), dtype=float)
        f2_raw = np.zeros(len(X), dtype=float)
        f3_raw = np.zeros(len(X), dtype=float)

        for i, individuo in enumerate(X):
            individuo = np.round(individuo).astype(int)
            estaciones_osmid = self.gdf["osmid"].iloc[np.where(individuo == 1)[0]].tolist()

            # --- Caso sin estaciones ---
            if len(estaciones_osmid) == 0:
                f1_raw[i] = 1.0
                f2_raw[i] = self.d_max * self.penalizacion
                f3_raw[i] = 1.0
                if self.debug:
                    print(f"[{i}] ⚠️ Individuo vacío → penalización total")
                continue

            # --- Objetivo 1: cantidad de estaciones ---
            f1_raw[i] = fitness_cantidad_estaciones(estaciones_osmid, self.factor_estaciones)

            estaciones_idx = np.where(individuo == 1)[0]

            f2_raw[i] = fitness_cobertura_global(
                self.gdf, estaciones_idx,
                d_max=self.d_max,
                penalizacion=self.penalizacion
            )

            # --- Objetivo 3: prioridad (flujo + esquinas) ---
            f3_raw[i] = 1 - fitness_prioridad_simple(
                self.gdf, estaciones_osmid,
                peso_flujo=self.peso_flujo,
                peso_esquinas=self.peso_esquinas
            )

            if self.debug and i < 5:
                print(f"[{i}] Estaciones={len(estaciones_osmid)}, F1={f1_raw[i]:.3f}, "
                      f"F2={f2_raw[i]:.1f}, F3={f3_raw[i]:.3f}")

        # --- Normalización / salida ---
        out["F"] = np.column_stack([
            f1_raw * 10, # penalización mucho mayor, da prioridad al numero de estaciones
            f2_raw / (self.d_max * self.penalizacion),  # normaliza cobertura
            f3_raw
        ])


In [106]:
def generar_poblacion_inicial(n_individuos, n_nodos, min_estaciones=1, max_estaciones=None):
    if max_estaciones is None:
        max_estaciones = n_nodos // 2
    X = np.zeros((n_individuos, n_nodos), dtype=int)
    for i in range(n_individuos):
        n_est = np.random.randint(min_estaciones, max_estaciones + 1)
        idx = np.random.choice(n_nodos, n_est, replace=False)
        X[i, idx] = 1
    return X

gdf = gdf_nodos  # usar el dataset limpio
d_max = 10000
pop_size = 200

algorithm = NSGA2(pop_size=pop_size, eliminate_duplicates=True)

X_init = generar_poblacion_inicial(pop_size, len(gdf), min_estaciones=1, max_estaciones=300)

res = minimize(
    EVChargingProblem3F(gdf, d_max=d_max),
    algorithm,
    ('n_gen', 1000),
    seed=42,
    verbose=True,
    save_history=True,
    X=X_init
)

print("Frontera de Pareto (f1: estaciones, f2: distancia, f3: calidad invertida):")
print(res.F)

n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |      200 |     34 |             - |             -
     2 |      400 |     46 |  0.1417450361 |         ideal
     3 |      600 |     56 |  0.0625000000 |         ideal
     4 |      800 |     62 |  0.0361926975 |         ideal
     5 |     1000 |     77 |  0.1248587070 |         ideal
     6 |     1200 |     92 |  0.0854767701 |         ideal
     7 |     1400 |    120 |  0.1294117647 |         ideal
     8 |     1600 |    100 |  0.0707413048 |         nadir
     9 |     1800 |    105 |  0.0317822441 |         ideal
    10 |     2000 |    124 |  0.0355587749 |         ideal
    11 |     2200 |    118 |  0.0173185486 |         ideal
    12 |     2400 |    124 |  0.1006417550 |         ideal
    13 |     2600 |    123 |  0.0808080808 |         ideal
    14 |     2800 |    132 |  0.0192164725 |         ideal
    15 |     3000 |    152 |  0.0557701004 |         ideal
    16 |     3200 |    153 |  0.0139373551 |         ide

In [107]:
import plotly.graph_objects as go
import numpy as np

# --- Supongamos que tu matriz de resultados es:
# res.F  → matriz (n_solutions, 3)
# columnas: [f1: estaciones, f2: distancia, f3: calidad invertida]

F = np.array(res.F)  # Asegurarse de que esté en formato numpy array

# Crear figura 3D interactiva
fig = go.Figure(data=[go.Scatter3d(
    x=F[:, 0],  # f1
    y=F[:, 1],  # f2
    z=F[:, 2],  # f3
    mode='markers',
    marker=dict(
        size=6,
        color=F[:, 2],          # color según calidad invertida
        colorscale='Viridis',   # escala de color bonita
        opacity=0.9,
        colorbar=dict(title="Calidad invertida")
    ),
    text=[f"Estaciones: {f1:.3f}<br>Distancia: {f2:.3f}<br>Calidad Inv: {f3:.3f}"
          for f1, f2, f3 in F],
    hoverinfo='text'
)])

# Configurar layout del gráfico
fig.update_layout(
    title="🌈 Frente de Pareto 3D — Estaciones vs Distancia vs Calidad",
    scene=dict(
        xaxis_title="f1: Estaciones (normalizado)",
        yaxis_title="f2: Distancia (normalizada)",
        zaxis_title="f3: Calidad invertida",
    ),
    width=900,
    height=700,
)

fig.show()


In [ ]:
import numpy as np
import geopandas as gpd


def interpretar_y_graficar_resultados(problem, X, F, gdf, n_mostrar=2):
    """
    Convierte los valores normalizados de F en métricas reales y
    grafica algunas soluciones sobre el mapa.

    Parámetros:
    ------------
    problem : EVChargingProblem3F
        Instancia del problema usado en el optimizador.
    X : np.ndarray
        Matriz de soluciones binarias (individuos).
    F : np.ndarray
        Matriz de valores de fitness (f1, f2, f3 normalizados).
    gdf : geopandas.GeoDataFrame
        Nodos con geometría (debe incluir columna 'osmid').
    n_mostrar : int
        Número de soluciones a graficar (por defecto 2).
    """

    resultados = []

    for i in range(len(X)):
        individuo = np.round(X[i]).astype(int)
        estaciones_idx = np.where(individuo == 1)[0]
        estaciones_osmid = gdf.iloc[estaciones_idx]["osmid"].tolist()

        # --- Desnormalizar valores ---
        f1_real = F[i, 0] / 10.0                      # volver a proporción original
        num_estaciones = int(round(f1_real * len(gdf)))
        f2_real = F[i, 1] * (problem.d_max * problem.penalizacion)  # distancia total real
        f3_real = 1 - F[i, 2]                         # prioridad real

        resultados.append({
            "indice": i,
            "num_estaciones": num_estaciones,
            "distancia_total": f2_real,
            "prioridad_media": f3_real,
            "estaciones_osmid": estaciones_osmid
        })

    df_resultados = gpd.pd.DataFrame(resultados)

    print("=== Resultados reales ===")
    print(df_resultados[["indice", "num_estaciones", "distancia_total", "prioridad_media"]].head(n_mostrar))

    # --- Graficar ---
    fig, axes = plt.subplots(1, n_mostrar, figsize=(7 * n_mostrar, 7))
    if n_mostrar == 1:
        axes = [axes]

    for ax, (_, fila) in zip(axes, df_resultados.head(n_mostrar).iterrows()):
        gdf_plot = gdf.copy()
        gdf_plot["estacion"] = gdf_plot["osmid"].isin(fila["estaciones_osmid"])

        gdf_plot.plot(ax=ax, color="lightgray", markersize=5, label="Nodos")
        gdf_plot[gdf_plot["estacion"]].plot(ax=ax, color="red", markersize=20, label="Estaciones")

        ax.set_title(
            f"Solución {fila['indice']}\n"
            f"Estaciones: {fila['num_estaciones']} | "
            f"Distancia: {fila['distancia_total']:.0f} | "
            f"Prioridad: {fila['prioridad_media']:.2f}",
            fontsize=11
        )
        ax.legend()
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    return df_resultados
